# LLM Fine-Tuning Deep Dive, Part 3 of 3: Comparison & Decision

> **Learning objective:** build intuition for how model evaluation matures from “this generated example looks promising” into evidence strong enough to support a workload decision.

Parts 1 and 2 produced the models. Part 3 asks one question about them: **what would make a claimed improvement believable enough to act on?**

## The Evaluation Story: Let the Evidence Evolve with the Objective

A plausible Riverside continuation starts a hypothesis; it does not prove one. This notebook begins with the simplest objective, continued pretraining, and develops the token-level score that objective makes meaningful. Only then does it ask why that same score cannot settle every question about SFT, DPO, parameter strategy, or release readiness.

![Six progressively stronger layers of evaluation evidence from an interesting output to a deployment decision](images/evaluation-evidence-ladder.png)

Examples on the left generate hypotheses; the evidence on the right supports decisions. Each step adds a necessary constraint, but no evaluation removes every risk.

## The Candidate Cast: Two Independent Fine-Tuning Choices

Parts 1 and 2 varied two different decisions:

- **Behavior objective:** what should the model learn to do differently?
- **Parameter strategy:** where should that learning be stored, and how much of the model should change?

```mermaid
flowchart LR
    subgraph Behavior["Behavior objective — what should be learned?"]
        direction TB
        B0["Base model<br/>general language"]
        B1["Continued pretraining<br/>domain language and style"]
        B2["SFT<br/>instruction contract"]
        B3["DPO<br/>relative editor preference"]
        B0 --> B1 --> B2 --> B3
    end

    subgraph Parameters["Parameter strategy — where is learning stored?"]
        direction TB
        P0["Full fine-tuning<br/>all weights"]
        P1["Partial freezing<br/>selected layers"]
        P2["LoRA<br/>small adapters"]
        P3["QLoRA<br/>quantized frozen base + adapters"]
        P0 --> P1 --> P2 --> P3
    end
```

The tracks are parallel, not one mandatory pipeline. In **SFT + LoRA**, SFT specifies the behavior being taught while LoRA specifies how the update is represented. Changing the parameter strategy does not change what behavioral success means.

## A Mental Model for Fine-Tuning Evaluation

Evaluation is not one score or a universal leaderboard. It connects training to an observable change, then asks whether that change is useful enough for a particular workload:

```mermaid
flowchart LR
    O["Training objective<br/>What was taught?"] --> C["Claim<br/>What should improve?"]
    C --> T["Observable evidence<br/>What would improvement look like?"]
    T --> M["Measurement<br/>How will examples be summarized?"]
    M --> D["Decision rule<br/>How much is enough?"]
    D --> R["Remaining risk<br/>What might the test miss?"]
```

A break anywhere weakens the conclusion. A test of next-token prediction cannot establish instruction following. A preference comparison is unconvincing if its test examples or judges leak information from training. Even strong task behavior is insufficient when safety, latency, or cost violates the deployment contract.

## Evaluation Roadmap

Read the notebook as an evaluation evolution, not as a universal scorecard:

| Evaluation evolution | Question that comes next | Evidence that becomes primary |
| --- | --- | --- |
| 1. Continued pretraining / full FT | Did Riverside prose become less surprising to the model? | Teacher-forced actual-token likelihood, then held-out corpus NLL/perplexity |
| 2. Comparability boundary | When do those values describe the same test, and when do they not? | Matched tokenizer, serialization, reference text, split, and claim |
| 3. SFT | Did the assistant follow the requested instruction? | Predeclared task contracts and pass/fail results |
| 4. DPO | Do editors prefer the DPO answer to the SFT answer? | Blinded pairwise preferences, ties, and uncertainty |
| 5. Parameter strategy | Does a cheaper update preserve the behavior we need? | The objective's behavior metric plus measured resource cost under a matched study |
| 6. Workload and release | Is the candidate useful and safe enough to operate? | Workload-specific quality, safety, latency, cost, lineage, and rollback gates |

---

## Setup: Reloading All Six Trained Checkpoints

Parts 1 and 2 ran in separate kernels and saved candidates under `./checkpoints/`, so this notebook reloads fresh Python objects rather than relying on hidden state.

> **Prerequisite:** Rerun Parts 1 and 2 from clean kernels with `HuggingFaceTB/SmolLM2-135M-Instruct` before running this notebook. Artifacts from other architectures cannot be reloaded here.

The six candidates are evidence sources, not one chronological checkpoint chain. Some differ in objective, parameter strategy, corpus, and hyperparameters; Part 3 will show why their score differences cannot be assigned to one training choice without a fairer comparison.

## The Core: One Instrument, Then Objective-Specific Evidence

Fine-tuning changes a model's conditional distribution, but people use the resulting behavior. The notebook therefore starts by learning one scoring instrument in the context where it is closest to the training objective: continued pretraining. It then changes the evidence when the objective changes.

| Objective | What training changes | First useful question | Why likelihood alone is or is not enough |
| --- | --- | --- | --- |
| Continued pretraining | Expectations for domain prose | Does it assign more probability to held-out Riverside text? | This is closely aligned with next-token training, so held-out NLL/perplexity is core evidence of prose fit |
| SFT | Response behavior under an instruction | Does it meet the requested contract? | A reference-answer likelihood is diagnostic, but many valid answers can satisfy one instruction |
| DPO | Relative preference between plausible responses | Do blinded editors prefer it to the SFT reference? | Pair likelihood movement is diagnostic, but preference must be observed in generated answers |
| Parameter strategy | Where the update is stored | Can it retain the target behavior at acceptable cost? | Reuse the behavior evidence for the chosen objective, then add resource measurements |

### The Learning Flow

The objective progression supplies the narrative:

1. **Learn the token-level instrument through continued pretraining:** observe a sample, remove decoding randomness with fixed text, and broaden from one phrase to a corpus.
2. **Find the instrument's boundary:** raw likelihood values are comparable only under a shared scoring contract and do not automatically represent every desired behavior.
3. **Change the primary evidence with the objective:** SFT needs instruction-contract success; DPO needs blinded preference; parameter comparisons need matched behavior and cost measurements.
4. **Use evidence for a real decision:** a workload selects the metric bundle, and release adds safety, operational, lineage, and rollback requirements.

This notebook is deliberately **retrospective**. The checkpoints already exist, so it investigates their apparent behavior before exposing which comparisons are descriptive only. In a prospective study, the data split, scoring contract, and matched-study controls would be fixed before training begins.

The walking sentence carries the first evolution: sampled appearance, fixed-text probability, then coverage across prose. Later sections do not promote that prose score into a universal winner; they ask what different evidence SFT, DPO, parameter strategy, and deployment claims actually need.

### Candidate Manifest: What Will Be Reloaded?

The notebook needs six independent model objects so that loading one adapter cannot mutate another candidate's base model.

| Candidate | Saved artifact | Objective | Parameter strategy | Ancestry |
| --- | --- | --- | --- | --- |
| Baseline | Hugging Face base checkpoint | Original pretraining | No Riverside update | SmolLM2 base |
| Full-FT continuation | `non-instruction-full` | Continued pretraining | Full fine-tuning | SmolLM2 base |
| Partial-freeze continuation | `partial-freeze` | Continued pretraining | Selected late layers | SmolLM2 base |
| LoRA continuation | `peft-lora` | Continued pretraining | LoRA | Fresh SmolLM2 base + adapter |
| SFT assistant | `instruction-lora` | SFT | LoRA | Fresh SmolLM2 base + adapter |
| DPO assistant | `preference-dpo` | DPO | Continue SFT LoRA | Fresh SmolLM2 base + DPO adapter |

The code first restores the common tokenizer and prompt contract, then loads a fresh base for every PEFT adapter, verifies the LoRA target modules, and derives parameter counts from the objects actually loaded. This reconstructs the experiment before any comparison begins.

> **PyTorch → Keras:** `torch.cuda.is_available()` + `.to(device)` explicitly move a model/tensors to
> GPU or CPU, `AutoModelForCausalLM.from_pretrained(...)` loads pretrained weights, and
> `model.generate(...)` run inside `torch.no_grad()` performs autoregressive decoding without tracking
> gradients (nothing to backprop through during inference). **Keras/TF equivalent:** TensorFlow places
> ops on GPU automatically (explicit placement is `tf.device(...)`, rarely needed); the loading call
> would be `TFAutoModelForCausalLM.from_pretrained(...)` followed by the same `.generate(...)` method --
> Keras/TF has no separate "no_grad" context since inference doesn't build a gradient tape by default.


In [ ]:
# Re-establish Parts 1-2's foundations using the same SmolLM2 base and prompt contract.
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
SYSTEM_PROMPT = "You are a careful fiction-writing assistant for Riverside Publishing."
CONTINUATION_INSTRUCTION = "Continue the fiction narrative in the same style."

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
PROMPT = "Aria Voss stared at the signal counting itself out in prime numbers and"

num_hidden_layers = base_model.config.num_hidden_layers
hidden_size = base_model.config.hidden_size
total_base_parameters = sum(parameter.numel() for parameter in base_model.parameters())
print(
    f"Loaded {MODEL_NAME}: {total_base_parameters:,} parameters, "
    f"{num_hidden_layers} decoder layers, hidden size {hidden_size}."
)


def instruction_prompt(prompt):
    """Build the user message used by the SFT and DPO recipes in Parts 1-2."""
    return f"{CONTINUATION_INSTRUCTION}\n\n{prompt}"


def apply_instruction_template(prompt):
    """Serialize an instruction through the model's native chat template."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def generate(model, prompt, max_new_tokens=60, use_chat_template=False):
    """Generate only new tokens, using the native chat format for instruction candidates."""
    model.eval()
    model_input = apply_instruction_template(prompt) if use_chat_template else prompt
    inputs = tokenizer(model_input, return_tensors="pt").to(device)
    prompt_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    completion = tokenizer.decode(
        output_ids[0][prompt_length:], skip_special_tokens=True
    ).strip()
    return completion if completion else "[model stopped immediately after the prompt]"


print(f"Baseline completion (sanity check): {generate(base_model, PROMPT)}")


### Reloading the Five Fine-Tuned Checkpoints

Each PEFT-wrapped adapter (instruction-tuned LoRA, DPO, LoRA continued pretraining) gets its own fresh
base-model instance rather than sharing `base_model` above -- the same "every PEFT wrapper gets its
own base" rule Parts 1-2 followed throughout. `freeze_model`'s `requires_grad` flags are re-applied
after loading (see the comment below) since that bookkeeping isn't part of a saved checkpoint -- only
the trained weights are.


> **PyTorch → Keras:** `PeftModel.from_pretrained(base_model, path)` wraps a fresh base model with a
> saved LoRA adapter's weights; `named_parameters()` iterates `(name, tensor)` pairs so `requires_grad`
> can be toggled per-parameter (used here to re-apply the freeze pattern, since that bookkeeping isn't
> part of a saved checkpoint), and `p.numel()` counts a tensor's elements to total trainable params.
> **Keras/TF equivalent:** LoRA loading has no single standard TF API (usually a custom `tf.keras.Model`
> subclass or a TF-specific PEFT integration); freezing is coarser-grained -- `layer.trainable = False`
> per layer rather than per-parameter -- and element counts come from `tf.size(variable)`.


In [ ]:
# Reload only artifacts regenerated by Parts 1-2 for MODEL_NAME.
non_instruct_ckpt = AutoModelForCausalLM.from_pretrained(
    "./checkpoints/non-instruction-full"
).to(device)

instruct_base_reload = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
instruct_lora_model = PeftModel.from_pretrained(
    instruct_base_reload, "./checkpoints/instruction-lora"
).to(device)

dpo_base_reload = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
policy_model = PeftModel.from_pretrained(
    dpo_base_reload, "./checkpoints/preference-dpo"
).to(device)

freeze_model = AutoModelForCausalLM.from_pretrained(
    "./checkpoints/partial-freeze"
).to(device)
n_layers = freeze_model.config.num_hidden_layers
unfreeze_from = n_layers - max(2, n_layers // 4)

for parameter in freeze_model.parameters():
    parameter.requires_grad = False
for layer in freeze_model.model.layers[unfreeze_from:]:
    for parameter in layer.parameters():
        parameter.requires_grad = True
for parameter in freeze_model.model.norm.parameters():
    parameter.requires_grad = True
for parameter in freeze_model.lm_head.parameters():
    parameter.requires_grad = True

lora_pt_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
lora_pt_model = PeftModel.from_pretrained(
    lora_pt_base, "./checkpoints/peft-lora"
).to(device)

adapter_models = {
    "instruction LoRA": instruct_lora_model,
    "DPO policy": policy_model,
    "continued-pretraining LoRA": lora_pt_model,
}
expected_targets = set(LORA_TARGET_MODULES)
for adapter_name, adapter_model in adapter_models.items():
    configured_targets = {
        target
        for peft_config in adapter_model.peft_config.values()
        for target in peft_config.target_modules
    }
    if configured_targets != expected_targets:
        raise ValueError(
            f"{adapter_name} targets {sorted(configured_targets)}, expected "
            f"{sorted(expected_targets)}. Rerun Parts 1-2 with the SmolLM2 LoRA recipe."
        )

for model in (
    non_instruct_ckpt,
    instruct_lora_model,
    policy_model,
    freeze_model,
    lora_pt_model,
):
    model.eval()

# Verify that the reconstructed partial-freeze model matches the intended layer policy.
trainable_partial_names = [
    name for name, parameter in freeze_model.named_parameters() if parameter.requires_grad
]
assert any(
    name.startswith(f"model.layers.{unfreeze_from}.")
    for name in trainable_partial_names
), "Expected SmolLM2 trailing-layer parameter names were not found"

print("Reloaded all six candidates (baseline + 5 fine-tuned).")
print(f"SmolLM2 layers/hidden size: {n_layers} / {freeze_model.config.hidden_size}")


---

## Comparing the Candidates Without Confusing the Axes

Before comparing outputs, place every candidate on the two tracks introduced above:

| Candidate | Behavior objective | Parameter strategy | Limitation it was created to address |
| --- | --- | --- | --- |
| Baseline | Original pretraining only | No Riverside update | Control: capable language model, but no Riverside adaptation |
| Continued pretraining | Next-token prediction on Riverside prose | Full fine-tuning | Learn catalog language and house style |
| Partial-freeze continuation | Next-token prediction on Riverside prose | Update selected late layers | Test whether less trainable state can carry domain adaptation |
| LoRA continuation | Next-token prediction on Riverside prose | Low-rank adapters | Store domain adaptation in a small swappable artifact |
| SFT LoRA | Supervised prompt/completion loss | Low-rank adapters | Teach direct instruction following and response format |
| DPO adapter | Chosen/rejected preference loss after SFT | Continue updating the SFT adapter | Prefer editor-ranked responses among plausible answers |

### Two comparisons that answer different questions

**Behavior progression:** baseline → continued pretraining → SFT → DPO asks whether each objective addresses a new workload limitation. Because each objective teaches different behavior, each checkpoint must be evaluated against the limitation it was trained to address.

**Parameter progression:** full fine-tuning → partial freezing → LoRA asks how much model state must change while pursuing the **same** behavior. A fair comparison changes only the update strategy while holding the model, data, training budget, randomness, and evaluation conditions fixed.

The current full, partial-freeze, and LoRA runs are useful demonstrations, but they are **not** that fair comparison because their data and hyperparameters differ. Any observed gap mixes parameter strategy with those other differences.

### What is absent from the experiment?

The complete design space has three objectives by three parameter strategies. Five combinations were trained. SFT with full fine-tuning or partial freezing and DPO with full fine-tuning or partial freezing remain unmeasured. QLoRA was explored as a memory-scaling mechanism in Part 2, not trained as another quality candidate.

Blank combinations mean **not measured**, not failed.

## Continued Pretraining Evaluation: Build the Scoring Instrument

### Start with a Full-FT Question: Complete One Riverside Sentence

One sentence will carry the evaluation story from sample generation to release reasoning:

| Role | Text |
| --- | --- |
| Shared prompt | `Aria Voss checked the Meridian's Promise status panel and` |
| Riverside continuation to score | ` opened the Keeper's maintenance logs` |
| Generic control continuation | ` looked at the screen` |

The working hypothesis is deliberately narrow:

> Continued pretraining made the Riverside continuation less surprising than it was to the base model, and changed it more selectively than the generic control.

This sentence is the springboard for the first objective. We will move from a visible sample to a fixed-text score, then from that one phrase to many Riverside passages. The later SFT and DPO sections will return to Aria with different questions and different primary evidence.

Before formalizing the comparison, inspect what changed. Generated examples are useful because failures are concrete: the model may use generic prose, ignore an instruction, violate a format, invent a catalog fact, or stop badly. These observations reveal the behavior that a later benchmark must test.

But examples answer only *what can happen*, not *how often it happens*. Generation is affected by prompt wording, chat formatting, temperature, seed, and output length. A compelling sample is therefore a hypothesis generator, not a score.

Read each output through three lenses:

1. **Domain behavior:** does it use Riverside-specific entities and relationships coherently rather than merely echoing a name?
2. **Task behavior:** does it continue prose or answer an instruction in the requested format and stop appropriately?
3. **Preference behavior:** does DPO differ consistently from SFT, or is one nicer-sounding sample just decoding randomness?

> **Predict:** Which failure will be easiest to see but hardest to quantify: generic style, instruction non-compliance, or editorial preference? Record one observable sign before running the comparison.

The next code cell is a demonstration of observation, not an evaluation harness. One sampled generation cannot support a ranking. That limitation motivates a fixed-text comparison: hold the walking problem's two continuations still and compare what the models expected to come next.

### Side-by-Side: Every Checkpoint on the Same Prompt

### Begin with the Simplest Training Question

Start with the most direct change in this collection: continued pretraining with full fine-tuning. Riverside fed the model prose and asked it to keep predicting the next token. The first evaluation question should therefore be equally direct:

> Given a Riverside prose prefix, did full fine-tuning make the held-out continuation less surprising than it was to the base model?

A generated sample can make that question feel concrete, but it cannot answer it reliably. Sampling settings and one early random choice can change the whole paragraph. We need a way to hold the prompt and desired continuation still, then inspect what probability the model assigns to the tokens that actually occur.

The next sections build that scoring instrument from the Aria walking sentence: next-token probability, teacher forcing, mean log-probability, NLL, and perplexity. First it will answer the continued-pretraining question. Only after we understand both its strength and its limits will we return to SFT and DPO and ask what additional evidence their different objectives require.

The other candidates remain visible in the sample comparison because they reveal different failure modes. They are not yet being ranked by the prose metric. That comparison would come before the metric has earned the right to answer their questions.

> **PyTorch → Keras:** `AutoModelForCausalLM.from_pretrained("./checkpoints/...")` reloads a saved
> fine-tuned checkpoint from disk into a fresh `torch.nn.Module`, then `.to(device)` places it on
> GPU/CPU before the loop below calls the `generate()` helper defined earlier on each model in turn.
> **Keras/TF equivalent:** `TFAutoModelForCausalLM.from_pretrained(path)` loads the same checkpoint
> format into a `tf.keras.Model`; TensorFlow doesn't need an explicit `.to(device)` call since device
> placement is handled by default device scoping (or `tf.distribute` for multi-device setups) instead.


In [ ]:
# Compare every candidate on shared catalog prompts.
COMPARISON_PROMPTS = {
    "scifi": "Aria Voss checked the Meridian's Promise status panel and",
    "fantasy": "Kerra Valmont felt all five tides simultaneously as",
    "mystery": "Elena Voss studied the 1879 survey map and realized",
    "cyberpunk": "In the Lower Stacks of Neo-Shanghai, Kai Chen",
}

models_to_test = {
    "Baseline (no fine-tuning)": base_model,
    "Continued pretraining (full FT)": non_instruct_ckpt,
    "Instruction-tuned (LoRA)": instruct_lora_model,
    "Preference-aligned (DPO)": policy_model,
    "Partial fine-tuning": freeze_model,
    "PEFT LoRA continued pretraining": lora_pt_model,
}

print("=" * 80)
print("QUALITATIVE CANDIDATE EXAMPLES - SAME PROMPTS, ONE SAMPLE EACH")
print("=" * 80)

for prompt_name, prompt in COMPARISON_PROMPTS.items():
    print(f'\nPrompt ({prompt_name}): "{prompt}"')
    print("-" * 80)

    for model_index, (model_name, model) in enumerate(models_to_test.items()):
        sample_seed = 42 + model_index
        torch.manual_seed(sample_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(sample_seed)

        uses_chat_template = "Instruction" in model_name or "Preference" in model_name
        effective_prompt = instruction_prompt(prompt) if uses_chat_template else prompt
        output = generate(
            model,
            effective_prompt,
            max_new_tokens=60,
            use_chat_template=uses_chat_template,
        )
        output_display = output[:120] + "..." if len(output) > 120 else output

        format_note = " + SmolLM2 chat template" if uses_chat_template else ""
        print(f"\n[{model_name}] seed={sample_seed}{format_note}")
        print(f"  Output: {output_display}")

print("\n" + "=" * 80)
print("READ THESE AS EXAMPLES:")
print("1. Catalog language: are names and setting details specific rather than generic?")
print("2. Task behavior: does the model continue prose or answer an instruction?")
print("3. Stability: a claim requires repeated prompts/samples and a scoring rubric.")
print("=" * 80)

### Inspect One Complete Output

The prompt matrix gives breadth but truncates each sample. Before leaving qualitative evidence, inspect one complete output from every candidate and expose the exact input format.

This matters because SFT and DPO learned through SmolLM2's chat template, while continuation models learned from plain prose. Sending every model the same raw string would test prompt mismatch as well as model behavior.

Use the same three lenses:

| Lens | Concrete sign to look for | Common false positive |
| --- | --- | --- |
| Domain language | Story-specific entities or relationships used coherently | Repeating a name copied from the prompt |
| Task behavior | Direct answer or bounded continuation in the requested format | Fluent prose that ignores the instruction |
| Preference signal | A repeatable difference between SFT and DPO | One nicer sample caused by decoding randomness |

Even the complete outputs remain sampled observations. The next step removes decoding randomness by scoring the same fixed phrases under both models.

In [ ]:
# Instruction and preference candidates use the chat contract from Parts 1-2.
instruct_prompt = instruction_prompt(PROMPT)
print(f"Shared prompt (plain models)       : {PROMPT!r}")
print(f"Shared user request (chat models) : {instruct_prompt!r}")
print()

print("=== Baseline (no fine-tuning) ===")
print(f"  Input : {PROMPT!r}")
print(f"  Output: {generate(base_model, PROMPT)}")
print()

print("=== Non-instructional continued pretraining (full fine-tune) ===")
print(f"  Input : {PROMPT!r}")
print(f"  Output: {generate(non_instruct_ckpt, PROMPT)}")
print()

print("=== Instruction-tuned (LoRA) ===")
print(f"  Input : SmolLM2 chat template over {instruct_prompt!r}")
print(
    f"  Output: {generate(instruct_lora_model, instruct_prompt, use_chat_template=True)}"
)
print()

print("=== Preference-aligned (DPO on the instruction-tuned adapter) ===")
print(f"  Input : SmolLM2 chat template over {instruct_prompt!r}")
print(f"  Output: {generate(policy_model, instruct_prompt, use_chat_template=True)}")
print()

print("=== Partial fine-tuning (layer freezing) ===")
print(f"  Input : {PROMPT!r}")
print(f"  Output: {generate(freeze_model, PROMPT)}")
print()

print("=== Parameter-efficient (LoRA continued pretraining) ===")
print(f"  Input : {PROMPT!r}")
print(f"  Output: {generate(lora_pt_model, PROMPT)}")
print()

### From One Sample to a Fixed-Text Comparison

The sampled outputs above were useful for seeing failures, but they mixed two causes:

- the model assigned expectations to possible next tokens;
- the decoder selected one path using sampling settings and a random seed.

Once one sampled token differs, every later token is conditioned on a different history. Comparing two completed paragraphs therefore cannot reveal whether a later difference came from training or from an earlier sampling fork.

The walking example gives us a fairer question. Keep the prompt and the known Riverside continuation fixed:

> After `Aria Voss checked the Meridian's Promise status panel and`, how strongly did each model expect ` opened`, then ` the`, then the later tokens, when every model saw the same true preceding text?

A model first emits unrestricted scores for all vocabulary choices. Only their relative size matters. Softmax converts those scores into one shared probability budget that sums to $1$, so “more expected” becomes a comparable share of that budget. The formula is less important here than the contract: same context, same possible tokens, probabilities summing to one.

For later positions, we keep supplying the true earlier tokens instead of a model's sampled mistakes. The next section names this procedure **teacher forcing**, but its purpose comes first: prevent paths from drifting so each position remains the same comparison for every model.

### Teacher Forcing: Hold the Text Fixed and Measure Probability Shift

**Full-FT question:** was an apparent improvement caused by changed model probabilities or by one lucky decoding path?

We now freeze the walking problem:

- prompt: `Aria Voss checked the Meridian's Promise status panel and`
- Riverside target: ` opened the Keeper's maintenance logs`
- generic control: ` looked at the screen`

### Foundation: what is a next-token probability space?

A language model does not produce the target continuation in one step. It repeats one operation: given all tokens seen so far, assign a score to **every token in its vocabulary** as the possible next token.

1. The tokenizer converts text into token IDs. A token may be a word, part of a word, punctuation, or whitespace-bearing fragment.
2. The model reads the current token sequence and emits one raw score, called a **logit**, for each vocabulary token.
3. Softmax converts those logits into nonnegative probabilities that sum to $1$.
4. One probability is attached to each possible next token. That complete vector is the **next-token probability distribution** for this exact context.

Immediately after the walking prompt, an illustrative distribution might look like this:

| Candidate next token | Conditional probability |
| --- | ---: |
| ` opened` | $0.50$ |
| ` looked` | $0.20$ |
| ` said` | $0.10$ |
| every other vocabulary token combined | $0.20$ |

These are alternatives at **one position**, so they sum to $1$. If ` opened` is appended, the context changes and the model computes a new distribution for the next position, perhaps assigning probability $0.20$ to ` the`. “Probability space” here is not a mysterious hidden geometry; it is the vocabulary alternatives and their context-dependent probability mass at each prediction step.

A high probability means “this token is expected under the model after this context.” It does not mean the token is true, safe, or best for the user.

### Generation versus scoring

A generated paragraph combines two systems:

```mermaid
flowchart LR
    P["Walking prompt tokens"] --> M["Model<br/>one probability per vocabulary token"]
    M --> S["Decoder chooses one token<br/>temperature, top-p, seed"]
    S --> C["Append chosen token<br/>build a new context"]
    C --> M
    S --> O["Eventually: one sampled completion"]
    M --> F["Alternative: gather each known target token<br/>to score the fixed continuation"]
```

The model supplies a distribution; the decoder selects from it. Greedy decoding picks the largest probability, while sampling can pick another token according to a temperature/top-$p$ policy. A new seed can therefore change the completion without changing the model's probabilities at all.

To compare models more directly, keep the prompt and target fixed. At each position, look up the probability assigned to the token that actually appears next. This is **teacher forcing**: when scoring token $w_t$, provide the real preceding target tokens $w_{<t}$ rather than feeding back a model-generated path. Every model is tested on exactly the same contexts and targets.

### Score One Fixed Continuation

To keep the arithmetic readable, consider the first two illustrative target tokens. The real tokenizer may divide the text differently; the next code cell prints its actual tokenization and scores every resulting token.

| Scoring step | Context supplied to model | Actual target token | Probability gathered |
| ---: | --- | --- | ---: |
| 1 | `... status panel and` | ` opened` | $0.50$ |
| 2 | `... status panel and opened` | ` the` | $0.20$ |

The probability assigned to this two-token prefix is the product of its conditional probabilities:

$$
p(\text{ opened the}\mid\text{walking prompt})
=0.50\times0.20=0.10.
$$

The full target adds probabilities for `Keeper`, `'s`, `maintenance`, `logs`, or whatever token pieces the tokenizer actually produces. Products across many tokens quickly become tiny. Logs turn multiplication into addition:

$$
\log 0.50+\log 0.20=-0.693-1.609=-2.303.
$$

Divide by the two predicted tokens to place this prefix on a per-token scale:

$$
\text{mean log-probability}
=\frac{-2.303}{2}=-1.151\ \text{nats/token}.
$$

For the complete continuation, average the log-probabilities assigned to every actual continuation token. A higher mean log-probability means that fixed continuation was less surprising to that model. Negative log-likelihood (NLL) flips the sign, so lower NLL means the same thing.

The continuation is reference text supplied for scoring; the metric does not decide that it is the uniquely correct answer.

### Compare the Model Changes

A mean log-probability is one score for **one model reading one fixed continuation**. The baseline comparison appears only when we put the same continuation under both models.

The values below are illustrative. The code calculates the same columns from the real checkpoints.

| Fixed continuation after the same prompt | Base model score | Adapted model score | Change after adaptation |
| --- | ---: | ---: | ---: |
| Riverside target: ` opened the Keeper's maintenance logs` | $-5.0$ | $-3.0$ | $+2.0$ |
| Generic control: ` looked at the screen` | $-2.0$ | $-1.8$ | $+0.2$ |

Read the matrix in this order:

1. **Stay within one row first.** For the Riverside phrase, the score moved from $-5.0$ to $-3.0$, so this fixed phrase became less surprising after adaptation.
2. **Repeat the same before/after comparison for the generic phrase.** It also became less surprising, but only by $+0.2$.
3. **Compare the two changes, not the raw scores.** The Riverside phrase gained $+2.0$, while the generic phrase gained $+0.2$. The Riverside phrase therefore gained $+1.8$ more.

Do **not** compare the raw adapted scores $-3.0$ and $-1.8$ directly. The generic phrase may simply have been easier for both models before training. The useful question is whether adaptation changed each phrase differently from its own starting point.

The short labels for those two subtractions are:

- phrase shift: $\Delta(c)=\text{adapted score for }c-\text{base score for }c$;
- selectivity contrast: $\Delta(\text{Riverside phrase})-\Delta(\text{generic phrase})$.

For the worked matrix, the selectivity contrast is $(+2.0)-(+0.2)=+1.8$. A positive result supports the narrow local hypothesis that adaptation favored this Riverside-specific phrase more than this plausible generic alternative.

The **Riverside target** is a deliberately domain-specific probe phrase, not “the one correct answer.” The **generic control** is a plausible continuation with no Riverside-specific content. It tells us whether the score change was broad rather than specifically tied to the Riverside material.

This still covers only the chosen phrases. It does not establish generalization, overall writing quality, factual correctness, or instruction following.

### State the Continued-Pretraining Claim Boundary

This fixed-text comparison follows only the continued-pretraining claim. It asks whether adaptation raised a selected Riverside phrase's score more than a selected generic phrase's score.

That is a useful local mechanism check. It does **not** establish overall style quality, factual correctness, generalization beyond the selected phrases, or instruction following. After broadening this prose score, the notebook will introduce the different evidence SFT and DPO require.

### Prediction before measurement

For the walking prompt, continued pretraining should raise the Riverside target's score more than it raises the generic control's score. If both row changes are equal, the selectivity contrast is $0$: adaptation may have changed scores, but this example gives no evidence that the change favored Riverside content specifically.

### Demonstration: score fixed continuations

The next code cell computes the same matrix from the real checkpoints: each model's score, the adapted-minus-base change for every phrase, and the Riverside-versus-generic selectivity contrast. This removes decoding randomness and tests one mechanism. It does **not** establish that generated claims are correct, instructions are followed, or the effect extends across the corpus.

That remaining coverage problem motivates the next step: ask whether the same lower-surprise pattern holds across many unseen Riverside examples. The [later confidence chapter](../05-llm-evaluation/04-calibration-and-confidence.ipynb) develops the separate question of whether a model's stated confidence matches observed correctness.

> **PyTorch → Keras:** `model.eval()` switches dropout/batch-norm-style layers to inference behavior;
> `torch.no_grad()` disables gradient tracking for the forward pass below; calling `model(**inputs)`
> runs a forward pass and returns `outputs.logits` (raw scores), and `F.softmax(logits, dim=-1)`
> (from `torch.nn.functional`) converts those logits into a probability distribution over the vocabulary.
> **Keras/TF equivalent:** Keras layers infer train/inference behavior automatically (or via a
> `training=False` argument) instead of an explicit `.eval()` call, and there's no separate "no_grad"
> context since plain forward calls outside a `GradientTape` don't track gradients; the softmax step is
> `tf.nn.softmax(logits, axis=-1)` -- same idea, `axis` instead of `dim`.


In [ ]:
# Trace one fixed sentence token by token, then compare complete continuation scores.
import math

import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F

plt.rcParams.update({"figure.dpi": 100, "font.size": 10})

prompt_for_analysis = "Aria Voss checked the Meridian's Promise status panel and"
walking_target_label = "Keeper's maintenance logs"
generic_control_label = "looked at the screen"

# The first phrase is the walking target; the remaining phrases test selectivity.
candidate_phrases = {
    walking_target_label: " opened the Keeper's maintenance logs",
    "quantum fold drive": " checked the quantum fold drive",
    "containment-field anomaly": " detected a containment-field anomaly",
    generic_control_label: " looked at the screen",
    "said nothing": " said nothing",
    "went back to work": " went back to work",
}
domain_labels = set(list(candidate_phrases)[:3])


def continuation_logprob(model, prompt, continuation):
    """Return sequence summaries and the actual-token trace for one continuation."""
    prompt_ids = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    ).input_ids.to(device)
    full_ids = tokenizer(
        prompt + continuation, return_tensors="pt", add_special_tokens=False
    ).input_ids.to(device)
    prompt_length = prompt_ids.shape[1]
    if not torch.equal(full_ids[:, :prompt_length], prompt_ids):
        raise ValueError("Prompt tokens are not a stable prefix of prompt + continuation")

    model.eval()
    with torch.no_grad():
        logits = model(full_ids).logits[:, :-1, :]
        log_probs = F.log_softmax(logits, dim=-1)

    # Position prompt_length-1 predicts the first continuation token.
    continuation_ids = full_ids[:, prompt_length:]
    continuation_log_probs = log_probs[
        0, prompt_length - 1 : full_ids.shape[1] - 1
    ].gather(1, continuation_ids[0].unsqueeze(1)).squeeze(1)

    token_ids = continuation_ids[0].tolist()
    token_log_probs = continuation_log_probs.tolist()
    token_pieces = tokenizer.convert_ids_to_tokens(token_ids)
    trace = [
        {
            "token_id": token_id,
            "token": token_piece,
            "probability": math.exp(token_log_probability),
            "log_probability": token_log_probability,
            "surprise": -token_log_probability,
        }
        for token_id, token_piece, token_log_probability in zip(
            token_ids, token_pieces, token_log_probs
        )
    ]

    return {
        "mean": continuation_log_probs.mean().item(),
        "sum": continuation_log_probs.sum().item(),
        "tokens": continuation_ids.shape[1],
        "trace": trace,
    }


phrase_results = []
walking_traces = None
for label, phrase in candidate_phrases.items():
    baseline_score = continuation_logprob(base_model, prompt_for_analysis, phrase)
    finetuned_score = continuation_logprob(
        non_instruct_ckpt, prompt_for_analysis, phrase
    )
    if label == walking_target_label:
        walking_traces = {
            "baseline": baseline_score["trace"],
            "finetuned": finetuned_score["trace"],
        }
    phrase_results.append(
        {
            "label": label,
            "kind": "catalog" if label in domain_labels else "generic control",
            "tokens": finetuned_score["tokens"],
            "baseline": baseline_score["mean"],
            "finetuned": finetuned_score["mean"],
            "delta": finetuned_score["mean"] - baseline_score["mean"],
        }
    )

assert walking_traces is not None
assert [item["token_id"] for item in walking_traces["baseline"]] == [
    item["token_id"] for item in walking_traces["finetuned"]
]

print("=== Walking problem: actual next-token trace ===")
print(f"Prompt: {prompt_for_analysis!r}")
print(f"Fixed target: {candidate_phrases[walking_target_label]!r}\n")
print(
    f"{'Step':>4} {'Tokenizer piece':22} {'Base p':>10} {'Adapted p':>10} "
    f"{'Base surprise':>14} {'Adapted surprise':>17}"
)
print("-" * 92)
for step, (baseline_token, finetuned_token) in enumerate(
    zip(walking_traces["baseline"], walking_traces["finetuned"]), start=1
):
    print(
        f"{step:>4} {baseline_token['token']!r:22} "
        f"{baseline_token['probability']:>10.5f} "
        f"{finetuned_token['probability']:>10.5f} "
        f"{baseline_token['surprise']:>14.3f} "
        f"{finetuned_token['surprise']:>17.3f}"
    )
print(
    "\nRead one row at a time: both models see the same true prefix; lower surprise "
    "means the model assigned more probability to that actual next token."
)

print("\n=== Complete fixed-continuation scores ===")
print(
    f"{'Phrase':30} {'Type':16} {'Tok':>3} {'Base':>9} "
    f"{'Adapted':>11} {'Adapted - base':>15}"
)
print("-" * 94)
for row in phrase_results:
    print(
        f"{row['label']:30} {row['kind']:16} {row['tokens']:>3} "
        f"{row['baseline']:>9.3f} {row['finetuned']:>11.3f} {row['delta']:>+15.3f}"
    )

labels = [row["label"] for row in phrase_results]
baseline_values = [row["baseline"] for row in phrase_results]
finetuned_values = [row["finetuned"] for row in phrase_results]
deltas = [row["delta"] for row in phrase_results]
positions = np.arange(len(labels))
width = 0.36

fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1.35, 1]})

axes[0].barh(
    positions - width / 2,
    baseline_values,
    height=width,
    label="Baseline",
    color="#4C78A8",
)
axes[0].barh(
    positions + width / 2,
    finetuned_values,
    height=width,
    label="Continued pretraining",
    color="#E45756",
)
axes[0].set_yticks(positions)
axes[0].set_yticklabels(labels)
axes[0].invert_yaxis()
axes[0].set_xlabel("Mean log-probability per token (higher = less surprising)")
axes[0].set_title("One-Model Scores for Fixed Text")
axes[0].legend()
axes[0].grid(axis="x", alpha=0.25)

delta_colors = ["#2A9D8F" if value >= 0 else "#D1495B" for value in deltas]
axes[1].barh(positions, deltas, color=delta_colors)
axes[1].set_yticks(positions)
axes[1].set_yticklabels(labels)
axes[1].invert_yaxis()
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_xlabel("Adapted minus base (nats/token)")
axes[1].set_title("Baseline Comparison for Each Phrase")
axes[1].grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.show()

catalog_deltas = [row["delta"] for row in phrase_results if row["kind"] == "catalog"]
control_deltas = [
    row["delta"] for row in phrase_results if row["kind"] == "generic control"
]
walking_delta = next(
    row["delta"] for row in phrase_results if row["label"] == walking_target_label
)
generic_control_delta = next(
    row["delta"] for row in phrase_results if row["label"] == generic_control_label
)
walking_selectivity_delta = walking_delta - generic_control_delta

print("\n=== Nested comparisons ===")
print(f"Walking target, adapted - base:       {walking_delta:+.3f} nats/token")
print(
    f"Named generic control, adapted - base: {generic_control_delta:+.3f} nats/token"
)
print(
    f"Selectivity contrast (target - control): {walking_selectivity_delta:+.3f} nats/token"
)
print(f"Mean catalog shift:                    {np.mean(catalog_deltas):+.3f} nats/token")
print(f"Mean all-generic-controls shift:        {np.mean(control_deltas):+.3f} nats/token")
print(
    "Interpretation: each phrase shift compares adapted with base. The selectivity "
    "contrast then asks whether the walking target shifted more than the named generic control."
)
print(
    "A positive selectivity contrast supports this local Riverside-specific mechanism "
    "hypothesis; it is not a general quality or corpus-level result."
)

### Scale the Same Score from One Sentence to Corpus Fit

**Next full-FT question:** did the Riverside continuation become less surprising because training changed the model's expectations more broadly, or did we choose one unusually favorable phrase?

### One Scoring Operation, Several Views

This section does **not** introduce several independent scoring mechanisms. It follows one operation from a single token to many passages:

| Name in this scoring arc | Role in the same operation | Is it a new metric? |
| --- | --- | --- |
| Vocabulary distribution | The model's full set of next-token probabilities after one context | No; this is the raw prediction space |
| Actual-token probability | The one probability teacher forcing gathers for the token that really occurred | No; this is one observation |
| Surprise | A transformation of that one probability that makes unlikely actual tokens matter more | No; this makes observations additive |
| Mean log-probability / mean NLL | The average of those token observations over a phrase or corpus | This is the core aggregate score; the two names differ only by sign |
| Perplexity | A friendlier rescaling of the same average surprise | No; it reports the same aggregate on a different scale |
| Corpus probe | Many frozen passages replacing one selected phrase | No; this broadens the **sample**, not the score |

Keep this arrow in mind:

> full vocabulary distribution → probability of the actual next token → surprise → average across tokens → perplexity

The walking example follows that arrow first for ` opened`, then for the complete ` opened the Keeper's maintenance logs` continuation, and finally for many Riverside passages.

### Start with the Whole Vocabulary

At every position, the model assigns probability to **every token in its vocabulary**. The total is always $1$; what changes is how concentrated or spread out that probability is.

After `Aria Voss checked the Meridian's Promise status panel and`, imagine the base and adapted models assign these probabilities. They are illustrative values; the scoring code later uses the real checkpoints.

| Possible next token | Base model | Adapted model |
| --- | ---: | ---: |
| ` opened` | $0.20$ | $0.80$ |
| ` looked` | $0.18$ | $0.04$ |
| ` said` | $0.10$ | $0.02$ |
| all other vocabulary tokens | $0.52$ | $0.14$ |

Both models distribute probability across the full vocabulary. The adapted model has concentrated much more probability on ` opened`, the token that actually occurs in the Riverside continuation. At this one position, it is less unsure about what comes next.

That is the intuition we now broaden: across many real Riverside tokens, did adaptation repeatedly place more probability on what actually happened?

### From One Actual Token to Perplexity

Teacher forcing follows the real Riverside continuation. At each position, it gathers **one number**: the probability assigned to the token that actually occurred.

For the first token in ` opened the Keeper's maintenance logs`, use the illustrative distribution above:

| Model | Probability assigned to the actual token ` opened` | What that means |
| --- | ---: | --- |
| Base model | $0.20$ | ` opened` was one possibility among many; the base was fairly unsure |
| Adapted model | $0.80$ | The adapted model strongly expected the actual Riverside token |

This is already a valid local comparison. The same prompt, tokenizer, and actual token are held fixed, so $0.80$ versus $0.20$ says the adapted model assigned more probability to what happened.

### Important Precision: What Perplexity Does and Does Not Score

Your “ground truth versus not-ground-truth” picture is a useful way to locate the relevant probability, with one important refinement:

- Perplexity uses only $p(\text{actual token})$ at each position.
- It does **not** separately add the probability of every wrong token.
- It also does **not** explicitly add one second term for $1-p(\text{actual token})$.

The rest of the vocabulary still matters because all probabilities must sum to $1$. Every alternative token competes with ` opened` for probability mass. But after softmax has created that distribution, the scoring step gathers only the actual token's entry.

These two distributions therefore make the **same** perplexity contribution for ` opened`:

| Distribution after the prompt | $p(\texttt{ opened})$ | How the remaining $0.20$ is arranged | Perplexity uses |
| --- | ---: | --- | ---: |
| A | $0.80$ | One alternative has all $0.20$ | $0.80$ |
| B | $0.80$ | Hundreds of alternatives share $0.20$ | $0.80$ |

Perplexity cannot distinguish A from B at this position. It is **not a direct measurement of the full distribution's spread**. A metric such as entropy would inspect every probability in the vocabulary distribution; perplexity instead asks how much mass landed on the token that reality supplied.

### Why Turn Actual-Token Probability into Surprise?

Across a full paragraph, the model produces one actual-token probability at every position. We need an average that treats a nearly certain correct prediction as a tiny event and a very unlikely actual token as a serious miss.

Information theory uses **surprise** for this transformation:

$$
\text{surprise of an actual token}=-\ln(p).
$$

You do not need to memorize the formula. Its behavior is the point:

| Probability assigned to the actual token | Surprise | Intuition |
| ---: | ---: | --- |
| $0.99$ | $0.01$ | Almost no surprise; the model was essentially ready for it |
| $0.80$ | $0.22$ | Low surprise; a strong expectation |
| $0.20$ | $1.61$ | Noticeable surprise; much probability went to alternatives |
| $0.001$ | $6.91$ | Major surprise; the observed token was almost ruled out |

The negative logarithm barely changes confident, predictable cases but sharply penalizes a low-probability miss. That is useful because one token the model considered nearly impossible should matter more than several routine predictions it handled confidently.

For the illustrative ` opened` token, the adapted model's surprise is $0.22$, while the base model's is $1.61$. The adapted model is not merely “a bit better”: it placed four times as much probability on the actual token, and the surprise transformation makes that difference visible on an additive scale.

### Why Not Just Compare Raw Mean Log-Probabilities?

We **can** compare raw mean log-probabilities when candidates use the same tokenizer, prompt format, and reference text. That is what the fixed-text comparison above does. A higher value, such as $-0.9$ rather than $-1.6$, means the observed continuation was less surprising.

The difficulty is readability: the values are negative, live on a log scale, and are hard to interpret as an everyday magnitude across a long document. Perplexity does not make the comparison more scientifically valid. It expresses the same average surprise on a friendlier scale.

For a two-token illustration with actual-token probabilities $0.80$ and $0.20$:

| Step | Actual-token probability | Surprise |
| ---: | ---: | ---: |
| ` opened` | $0.80$ | $0.22$ |
| ` the` | $0.20$ | $1.61$ |
| **Average** | | $0.92$ |

Exponentiating that average surprise gives a **perplexity** of about $2.5$. Read it as: across these two actual tokens, the model was as uncertain as if it had to choose among roughly $2.5$ equally plausible options at each step.

That effective-choice wording is an analogy, not a literal count of vocabulary tokens above a cutoff. Perplexity is still based only on the probabilities assigned to the tokens that actually occurred.

### Broaden the Walking Example

The fixed-text comparison asked whether adaptation gave more probability to the selected phrase ` opened the Keeper's maintenance logs` than the base model did. That phrase may be unusually favorable. The next cells repeat the **same teacher-forced actual-token scoring** across many Riverside passages instead of one hand-picked continuation.

The question becomes:

> Across these passages, does the adapted model generally find the observed Riverside tokens less surprising than the base model does?

The code intentionally uses one shared later-chapter sample for every candidate so you can watch the single-phrase calculation become a corpus aggregation. That executable bridge is useful; it shows what perplexity aggregates and supplies a concrete result for the later coverage discussion.

It is **not** valid model selection evidence. The checkpoints saw different training data, objectives, and hyperparameters; full fine-tuning also saw some sampled chapters. A lower bar can therefore reflect training exposure or another recipe difference, not a better parameter strategy or a production-ready model.

Read the output as: “how do these already-trained candidates fit this shared sample?” Do not read it as: “which candidate is best?” The code labels the result as descriptive and prints the same warning next to the values.

> **PyTorch → Keras:** passing `labels=enc["input_ids"]` into the model's forward call makes the
> Hugging Face model compute cross-entropy loss internally and return it as `out.loss` (a scalar
> tensor); `torch.no_grad()` skips gradient tracking since this is evaluation-only, and `.item()`
> pulls the plain Python float out of that 0-d tensor, which `math.exp(loss)` then turns into
> perplexity. **Keras/TF equivalent:** the TF counterpart model supports the same `labels=` convenience
> (`model(enc, labels=...)`), while plain Keras code would instead call
> `tf.keras.losses.SparseCategoricalCrossentropy()(y_true, logits)` and use `.numpy()` in place of
> `.item()` to extract the scalar.


In [ ]:
# Resolve the Riverside corpus only when the corpus-level metric first needs it.
try:
    notebook_dir = Path(__vsc_ipynb_file__).parent  # type: ignore[name-defined]
except NameError:
    try:
        notebook_dir = Path(__file__).parent
    except NameError:
        notebook_dir = Path.cwd()

CONTENT_DIR = notebook_dir / "content"
if not CONTENT_DIR.exists():
    fallback_content_dir = Path.cwd() / "learning" / "genai" / "04-llm" / "content"
    if fallback_content_dir.exists():
        CONTENT_DIR = fallback_content_dir

NOVELS = {
    "scifi": "the-weight-of-distant-light",
    "fantasy": "the-tidebound-accord",
    "mystery": "the-cartographers-cipher",
    "historical": "the-silk-merchants-daughter",
    "cyberpunk": "neural-drift",
    "horror": "the-hollow-beneath",
    "literary": "the-weight-of-tides",
}

if not CONTENT_DIR.exists():
    raise FileNotFoundError(
        f"Riverside content directory not found at {CONTENT_DIR.absolute()}. "
        "Run this notebook from the repository workspace."
    )

print(f"Corpus evaluation setup: {CONTENT_DIR.absolute()} ({len(NOVELS)} novels)")

In [ ]:
import math


# Use the same later-chapter sample for every candidate. This is a shared probe, not a clean holdout,
# because upstream training coverage differs and full FT saw some of these files.
def load_corpus_probe(start_chapter_index=10, chapters_per_novel=2, min_len=200):
    paragraphs = []
    source_files = []

    for alias, novel_dir in NOVELS.items():
        novel_path = CONTENT_DIR / novel_dir
        chapter_files = sorted(novel_path.glob("chapter-*.txt"))
        probe_files = chapter_files[
            start_chapter_index : start_chapter_index + chapters_per_novel
        ]
        source_files.extend(probe_files)

        for path in probe_files:
            text = path.read_text(encoding="utf-8")
            for paragraph in text.split("\n\n"):
                paragraph = paragraph.strip().replace("\n", " ")
                if len(paragraph) >= min_len:
                    paragraphs.append(paragraph)

    return paragraphs, source_files


probe_paragraphs, probe_files = load_corpus_probe()
print(
    f"Shared corpus probe: {len(probe_paragraphs)} paragraphs from "
    f"{len(probe_files)} later-chapter files."
)
print(
    "WARNING: this is descriptive, not held out. Upstream candidates used different "
    "training files, and full fine-tuning saw some probe chapters."
)


# Aggregate negative log-likelihood by evaluated token rather than averaging paragraph means.
def compute_corpus_probe(model, paragraphs, max_length=128):
    model.eval()
    total_nll = 0.0
    total_tokens = 0

    with torch.no_grad():
        for paragraph in paragraphs:
            encoded = tokenizer(
                paragraph,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            ).to(device)
            output = model(**encoded, labels=encoded["input_ids"])

            # Causal loss predicts tokens 2..N from tokens 1..N-1.
            valid_tokens = int(encoded["attention_mask"][:, 1:].sum().item())
            total_nll += output.loss.item() * valid_tokens
            total_tokens += valid_tokens

    mean_nll = total_nll / total_tokens
    return {
        "mean_nll": mean_nll,
        "perplexity": math.exp(mean_nll),
        "tokens": total_tokens,
    }


models_for_probe = {
    "Baseline (no fine-tuning)": base_model,
    "Full fine-tuning": non_instruct_ckpt,
    "Instruction-tuned (LoRA)": instruct_lora_model,
    "Preference-aligned (DPO)": policy_model,
    "Partial freezing": freeze_model,
    "LoRA continued pretraining": lora_pt_model,
}

print(f"\n{'=' * 72}\nShared corpus-probe perplexity (descriptive only):\n{'=' * 72}")
corpus_probe_results = {}
for name, model in models_for_probe.items():
    result = compute_corpus_probe(model, probe_paragraphs)
    corpus_probe_results[name] = result
    print(
        f"  {name:<32} NLL={result['mean_nll']:6.3f}  "
        f"PPL={result['perplexity']:8.1f}  tokens={result['tokens']:,}"
    )
print(f"{'=' * 72}")

probe_ranking = sorted(
    corpus_probe_results.items(), key=lambda item: item[1]["perplexity"]
)
names = [name for name, _ in probe_ranking]
perplexities = [result["perplexity"] for _, result in probe_ranking]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(names[::-1], perplexities[::-1], color="#4C78A8")
ax.set_xlabel("Corpus-probe perplexity (lower = better fit to this sample)")
ax.set_title(
    "Shared Later-Chapter Corpus Probe\n"
    "Descriptive only: training exposure differs across candidates",
    fontsize=11,
    fontweight="bold",
)
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

lowest_name, lowest_result = probe_ranking[0]
print(
    f"Lowest value on this contaminated probe: {lowest_name!r} "
    f"(PPL={lowest_result['perplexity']:.1f})."
)
print(
    "Do not promote that candidate from this ranking. A valid selection requires a split "
    "created before matched training plus workload-specific task metrics."
)

## Continued-Pretraining Checkpoint: What This Score Supports

Follow what happened to the original sentence problem:

1. A sampled completion suggested that Riverside adaptation might have changed model behavior.
2. Fixing ` opened the Keeper's maintenance logs` removed decoding luck and produced one mean log-probability under each model.
3. Subtracting the base-model score from the adapted-model score tested whether that same continuation became less surprising after adaptation.
4. Repeating that subtraction for ` looked at the screen`, then comparing the two shifts, tested whether the local probability change was more Riverside-selective than generic.
5. Extending adapted-versus-base scoring across frozen prose produced corpus NLL/perplexity and tested distribution fit more broadly.

Even a clean perplexity improvement would support only this claim:

> The adapted model assigns more probability than the base model to held-out Riverside token sequences under the declared tokenizer and context policy.

It would not establish instruction following, editor preference, factuality, safety, or which training choice caused the difference. The next question is therefore not immediately which checkpoint has the lowest number. First establish when a likelihood comparison means the same thing across candidates. Then decide what evidence the SFT and DPO objectives actually require.

## Decide the Evidence Before Looking at Scores

We can now name the scoring instrument we built through continued pretraining. Teacher forcing gives a causal language model a fixed input and gathers the probability it assigned to each actual response token. Mean log-probability, NLL, and perplexity are different views of that same calculation.

For continued pretraining, that calculation is close to the thing training optimized: how expected is held-out Riverside prose? It was therefore the right primary evidence for the first chapter. It is not automatically an apples-to-apples model ranking.

### Before comparing likelihood values, match the scoring contract

A raw NLL or perplexity value is comparable only when every candidate is asked the **same likelihood question**:

| Condition that must match | Why it matters |
| --- | --- |
| Tokenizer and model vocabulary | A different token split changes the unit being averaged |
| Input serialization | Plain prose and a chat template are different contexts, even when their visible words look similar |
| Fixed reference continuation | Different target text can be inherently easier or harder |
| Context length, truncation, and masking policy | These determine which tokens the model is allowed to condition on and which tokens count |
| Frozen evaluation split | Training exposure turns apparent generalization into familiarity |
| Behavior claim | A score can be calculated for many tasks without measuring the behavior a user needs |

The current collection violates several of these conditions across objectives. Continued-pretraining candidates consume plain Riverside prose. SFT and DPO consume an instruction serialized through the SmolLM2 chat template. They also learned different data and objectives. Their shared corpus-probe bars are therefore descriptive diagnostics, not a leaderboard that can declare one of the six candidates best.

### The calculation survives; the primary evidence changes

The same teacher-forced likelihood calculation is still useful after continued pretraining, but later it answers a narrower diagnostic question:

| Objective | Aria-version of the question | What likelihood can diagnose | What must be primary evidence |
| --- | --- | --- | --- |
| Continued pretraining | Does the model expect ` opened the Keeper's maintenance logs` in held-out prose? | Direct fit to the target prose distribution | Held-out corpus NLL/perplexity, then style and general-language checks |
| SFT | Does it obey `Continue the scene with exactly one sentence and stop`? | Whether a fixed reference response became more or less expected under the correct chat template | Predeclared instruction cases that pass or fail explicit requirements |
| DPO | Does an editor prefer its answer to the SFT answer? | Whether chosen responses moved above rejected responses relative to SFT | Blinded SFT-versus-DPO comparisons, recorded as win, loss, or tie |
| Parameter strategy | Can full FT, partial freezing, and LoRA preserve one target behavior? | The same objective-specific behavior score under a matched protocol | That behavior evidence plus memory, time, artifact, latency, and cost measurements |

SFT can have many valid answers to one instruction, so high likelihood for one reference answer cannot prove that the assistant followed the contract. DPO can move probability in the intended chosen-over-rejected direction, yet still produce answers editors do not prefer when generated. In both cases likelihood helps find regressions and explain model movement; it does not replace observing the behavior the objective promised.

The next two sections keep Aria in view, but they no longer ask the continued-pretraining question. They add the evidence needed for an instruction-following assistant and a preference-aligned assistant.

## SFT Evaluation: Does the Assistant Meet an Instruction Contract?

Continued pretraining asked whether a model fit Riverside prose. SFT asks a different question: can the model respond to an editor's request in the required form? The instruction and the response must be serialized through the chat template used in training.

Teacher-forced likelihood still has a role here. Score the response tokens only, under that chat contract, to spot a mismatch or a regression in reference matching. But a strong score for one reference response cannot prove instruction following: several responses may satisfy the same request, and a fluent response can still violate a length, format, source, or stopping requirement.

SFT evaluation therefore begins by defining cases that can visibly pass or fail before looking at a candidate's output.

### Run Aria Through One SFT Evaluation Case

Keep the same story, but change the input contract. The editor supplies a short source passage and asks: `Continue the scene with exactly one sentence. Keep Aria aboard the Meridian's Promise. Do not introduce unsupported facts.` The SFT candidate receives that request through its chat template and generates one answer.

That one response creates three kinds of evidence:

| Requirement for this Aria case | Who or what evaluates it? | Result recorded |
| --- | --- | --- |
| Exactly one sentence; clean stop; valid required format | Deterministic code | Pass or fail for each mechanical rule |
| No contradiction of the supplied passage | Deterministic source checks when the fact is explicit; otherwise an editor | Pass, fail, or needs review with a reason |
| Preserves Riverside voice and is useful to an editor | Qualified Riverside editor using a written rubric | Rubric score and short rationale |

A **judge** is therefore not automatically another model. Start with deterministic checks whenever the contract is precise. For subjective quality, the authority should be a qualified human editor who can see the source passage and a predeclared rubric. A separate judge model can help triage or scale review only after it has been calibrated against a human-labeled set, its agreement and failures are reported, and it cannot see the candidate's identity. It is a measurement instrument, not the definition of correctness.

For this case, Riverside can declare a final pass only when every required mechanical check passes and the required editorial review clears its threshold. Repeating that case pattern across a held-out suite turns individual pass/fail records into the SFT pass rate.

### Turn Instructions into Testable Contracts

An instruction benchmark needs more than a list of prompts. Each case should declare the behavior being tested, valid outputs, and a scorer before model outputs are inspected.

Once each case has an explicit pass/fail result, summarize the fraction that passed. This **pass rate** is

$$
\text{pass rate} = \frac{\sum_{i=1}^{N}\mathbb{1}[\text{case } i \text{ passes}]}{N}.
$$

The indicator may come from exact schema validation, a task-specific function, or a blinded rubric. Prefer deterministic scorers where the contract permits them. Use human or model judges only for qualities that cannot be reduced honestly to rules, and first compare their scores with human-labeled examples to reveal where they agree or fail.

A useful SFT suite separates failure types rather than hiding them in one average:

- instruction selection: did the model attempt the requested task?
- constraint adherence: did it respect format, length, tone, and prohibited content?
- task correctness: was the answer actually correct or supported by the provided sources?
- stopping behavior: did it end cleanly without continuing the dialogue for the user?
- robustness: does the contract survive paraphrases and difficult slices?

## DPO Evaluation: Is the Response Actually Preferred?

DPO is evaluated relative to a reference behavior, usually the SFT checkpoint. Its pair likelihoods are useful diagnostics: on held-out pairs, inspect whether chosen responses moved above rejected responses relative to SFT. That verifies the intended preference-learning direction, not whether generated answers are better for editors.

To test the behavior that matters, generate responses from SFT and DPO under matched decoding, hide model identity and response order, then ask qualified judges which response they prefer.

### Run Aria Through One Blind DPO Comparison

Use the same held-out Aria request and source passage from the SFT case. Generate one response from the accepted SFT checkpoint and one from the DPO checkpoint using the same decoding policy. Randomly label them **A** and **B**, then give an editor only the passage, the request, and the two anonymous responses.

The editor chooses `A`, `B`, or `tie` using a rubric such as: both answers must first satisfy the SFT contract; among eligible answers, which better preserves voice, advances the scene, and avoids unsupported claims? The editor does not know which answer came from DPO. Multiple independent editors can judge the same packet; disagreement is evidence about the uncertainty of the preference, not noise to hide.

Here the qualified Riverside editor is the preferred judge because the claim is an editorial preference. A calibrated judge model may screen obvious contract failures or provide a secondary, blinded rating at scale, but it must be checked against editor judgments, randomized for A/B position, and never treated as an unexamined substitute for them. If the requirement is entirely mechanical, deterministic code should decide it instead; that is a contract check, not the rich preference DPO is meant to learn.

After revealing the randomized labels, record whether DPO won, lost, or tied that one comparison. One choice is an observation, not a result. Repeating the blind packet across held-out requests produces the counts used in the win rate below.

Count DPO wins as $W$, losses as $L$, and ties as $T$. If a tie receives half credit, the resulting **win rate** is

$$
\text{win rate} = \frac{W + 0.5T}{W + L + T}.
$$

A value above 50% is not automatically convincing because a finite set of prompts and judgments can produce a noisy estimate. Report a **confidence interval**: a range that communicates how precisely this evaluation estimates preference under repeated sampling. Also slice by prompt type, randomize left/right order, inspect length bias, and measure judge agreement. Most importantly, rerun the SFT task suite: preference gain that breaks instruction following, safety, or diversity is a regression disguised as a win.

## Parameter-Strategy Evaluation: Can a Cheaper Update Preserve the Behavior?

Full FT, partial freezing, LoRA, and QLoRA are not different user goals. They are different ways to store the update. A fair strategy comparison therefore asks one simple question: **with the same training objective and experiment, does the cheaper update preserve the behavior we need?**

For example, compare two SFT runs on Aria's editing request. Keep the base model, instruction examples, train/validation/test split, chat template, optimizer policy, token budget, and seed set fixed. Train one run with full FT and one with LoRA. The parameter strategy is the only planned difference.

### Use Quality Metrics First, Then Operational Metrics

| Metric family | What it tells us about the candidate | How its value changes the decision |
| --- | --- | --- |
| **Primary behavior metric**: held-out perplexity for continuation, instruction pass rate for SFT, or blinded win rate for DPO | Did this strategy preserve the behavior its objective was meant to teach? | It is a **quality floor**. A candidate that misses the required behavior does not advance because it is cheaper. |
| **Regression and difficult-case checks**: source support, safety, format, robustness | Did the strategy quietly damage an important behavior hidden by the headline score? | A critical regression blocks promotion; inspect slices rather than averaging it away. |
| **Peak memory and training time** | Can the strategy be trained within the available hardware and delivery window? | Between candidates that clear the quality floor, lower resource use is an advantage. |
| **Artifact size, serving latency, and request cost** | Can the strategy be versioned and served within the production budget? | These are selection constraints, not proof of quality. A small adapter that misses latency or quality limits still loses. |

Interpret the values in that order. If LoRA and full FT both meet the SFT pass-rate, factuality, and safety requirements with no meaningful regression, prefer the lower-burden option. If full FT has a repeatable quality gain large enough to matter for the workload, Riverside may accept its higher cost. If the quality results are noisy or the experiment changed more than the strategy, do not claim either strategy won.

![Objective-specific training signals and the evidence needed for continued pretraining, SFT, and DPO](images/objective-specific-evaluation-evidence.png)

### Technique Combination Grid: Data x Parameter

The two independent choices become visible as a grid:

- A **row** fixes the behavior objective: continued pretraining, SFT, or DPO.
- A **column** fixes the parameter strategy: full fine-tuning, partial freezing, or LoRA.

| Data objective | Full fine-tuning | Partial freeze | LoRA |
| --- | --- | --- | --- |
| Continued pretraining | Trained: `non_instruct_ckpt` | Trained: `freeze_model` | Trained: `lora_pt_model` |
| Instruction tuning (SFT) | Not trained | Not trained | Trained: `instruct_lora_model` |
| Preference alignment (DPO) | Not trained | Not trained | Trained: `policy_model` |

Read the grid in two directions:

- **Across one row:** compare parameter strategies while holding the objective fixed. The result supports a cause claim only when data, hyperparameters, seeds, and evaluation are also matched.
- **Down one column:** compare objectives implemented with the same parameter strategy, but change the behavior evidence to match each objective.

Grey cells mean **not trained**, not failed. The sparse SFT and DPO rows explain why this notebook cannot compare their parameter strategies. The populated continued-pretraining row looks complete, but its upstream recipes were not matched, so it still cannot isolate the update strategy.

### Demonstration: visualize coverage and confounding

The next code cell shows both useful information and its boundary:

- Shared-probe perplexity describes fit to sampled prose.
- Trainable-parameter percentage describes update scope.
- Neither plot measures causal quality, task compliance, preference, peak memory, elapsed training time, serving latency, or cost.

The grid locates the missing experiment. The next section turns that gap into a one-factor-at-a-time study: vary one component, hold the rest fixed, and use the behavior evidence that matches the claim.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Derive update budgets where the parameter-strategy comparison first uses them.
total_params = sum(parameter.numel() for parameter in base_model.parameters())
full_ft_params = sum(parameter.numel() for parameter in non_instruct_ckpt.parameters())
partial_ft_params = sum(
    parameter.numel() for parameter in freeze_model.parameters() if parameter.requires_grad
)
lora_params = sum(
    parameter.numel()
    for name, parameter in instruct_lora_model.named_parameters()
    if ".lora_" in name
)
param_counts = [full_ft_params, partial_ft_params, lora_params]
param_pcts = [count / total_params * 100 for count in param_counts]

print("Update budgets for the matched-strategy question:")
print(f"  Full fine-tuning: {full_ft_params:,} parameters ({param_pcts[0]:.2f}%)")
print(f"  Partial freezing: {partial_ft_params:,} parameters ({param_pcts[1]:.2f}%)")
print(f"  LoRA matrices:    {lora_params:,} parameters ({param_pcts[2]:.2f}%)")

# Rows are learning objectives; columns are parameter strategies.
data_objectives = [
    "Continued\nPretraining",
    "Instruction\nTuning (SFT)",
    "Preference\nAlignment (DPO)",
]
parameter_strategies = [
    "Full FT\n(100%)",
    "Partial Freeze\n(~21%)",
    "LoRA\n(<1%)",
]

# Only five of the nine possible recipe cells were trained.
checkpoint_map = {
    (0, 0): "Full fine-tuning",
    (0, 1): "Partial freezing",
    (0, 2): "LoRA continued pretraining",
    (1, 2): "Instruction-tuned (LoRA)",
    (2, 2): "Preference-aligned (DPO)",
}
trained_mask = np.zeros((3, 3), dtype=bool)
probe_grid = np.full((3, 3), np.nan)
for (row, column), checkpoint_name in checkpoint_map.items():
    trained_mask[row, column] = True
    if checkpoint_name in corpus_probe_results:
        probe_grid[row, column] = corpus_probe_results[checkpoint_name]["perplexity"]

# Nominal training-time parameter budgets reconstructed earlier.
parameter_row = [param_pcts[0], param_pcts[1], param_pcts[2]]
parameter_grid = np.array([parameter_row] * 3)


def draw_recipe_grid(axis, values, value_format, title, color_map, legend_label):
    """Draw measured cells and leave untrained combinations visibly blank."""
    display_values = np.where(trained_mask, values, np.nan)
    color_map = plt.get_cmap(color_map).copy()
    color_map.set_bad(color="#D9D9D9")

    valid_values = display_values[trained_mask]
    value_min = valid_values.min()
    value_max = valid_values.max()
    if value_min == value_max:
        value_max = value_min + 1

    image = axis.imshow(
        display_values,
        cmap=color_map,
        vmin=value_min,
        vmax=value_max,
        aspect="auto",
    )

    for row in range(3):
        for column in range(3):
            if trained_mask[row, column]:
                axis.text(
                    column,
                    row,
                    value_format.format(display_values[row, column]),
                    ha="center",
                    va="center",
                    fontsize=11,
                    fontweight="bold",
                )
            else:
                axis.text(
                    column,
                    row,
                    "not\ntrained",
                    ha="center",
                    va="center",
                    fontsize=9,
                    color="#666666",
                    style="italic",
                )

    axis.set_xticks(range(3))
    axis.set_yticks(range(3))
    axis.set_xticklabels(parameter_strategies, fontsize=9)
    axis.set_yticklabels(data_objectives, fontsize=9)
    axis.set_xlabel("Parameter strategy", fontweight="bold")
    axis.set_ylabel("Data objective", fontweight="bold")
    axis.set_title(title, fontsize=11, fontweight="bold", pad=8)
    plt.colorbar(image, ax=axis, fraction=0.04, pad=0.04, label=legend_label)


fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(
    "Data Objective x Parameter Strategy: Coverage and Descriptive Signals",
    fontsize=13,
    fontweight="bold",
)

draw_recipe_grid(
    axes[0],
    probe_grid,
    "{:.1f}",
    "Shared Corpus-Probe Perplexity\n(confounded; not a model ranking)",
    "YlOrRd_r",
    "Probe perplexity",
)
draw_recipe_grid(
    axes[1],
    parameter_grid,
    "{:.2f}%",
    "Nominal Parameters Updated\n(not peak memory or latency)",
    "Blues_r",
    "Updated parameters (%)",
)

plt.tight_layout()
plt.show()

print("How to read the two panels:")
print("1. Topology: five recipe cells exist; four combinations were not trained.")
print("2. Left: probe values mix data exposure, objective, hyperparameters, and strategy.")
print("3. Right: parameter percentages describe update scope, not end-to-end resource cost.")
print("4. No quality/efficiency frontier can be inferred until the continued-pretraining row is retrained under a matched protocol.")

## Matched Ablation: Change One Thing

Your understanding is correct. An **ablation** keeps the experiment fixed and changes one planned variable, then asks how that change affects the metrics that matter for the objective.

### Aria SFT: Full FT Versus LoRA

For Riverside's one-sentence Aria editing task, train a full-FT SFT candidate and a LoRA SFT candidate on the same ordered examples. Keep the base model, chat template, train/validation/test split, optimizer policy, token budget, and evaluation code fixed. Repeat the pair across the same seed set. The only planned difference is **how the SFT update is represented**.

| Measure after training | What it answers | How Riverside uses the value |
| --- | --- | --- |
| Instruction pass rate, source support, and safety checks | Did each candidate still do the editing work correctly? | These are quality floors. A candidate that fails them is eliminated, even if it is faster or smaller. |
| Difficult-case and regression slices | Did one strategy fail on a subgroup that the overall average hides? | A critical regression blocks selection or triggers investigation. |
| Peak memory and training time | Can the candidate be trained within the available hardware and delivery window? | Use them to choose among candidates that clear the quality floor. |
| Artifact size, serving latency, and request cost | Can the candidate be deployed and operated within budget? | Treat them as production constraints and tie-breakers, never as substitutes for behavior quality. |

The decision is intentionally simple: choose LoRA when it matches full FT on the required quality and regression checks while reducing operational burden. Choose full FT only when its quality advantage repeats across seeds and matters enough to justify the added burden. If data, budget, or evaluation also changed, the observed difference is descriptive; rerun the matched study instead of declaring a winner.

![Confounded model comparisons contrasted with a controlled ablation that changes one factor at a time](images/controlled-comparison-vs-confounding.png)

The current checkpoint collection is not this ablation: it changed several data and training choices together. Its plots locate missing experiments, but cannot prove that a parameter strategy caused a quality difference.

Objective ablations use the same rule. To test whether SFT or DPO matters, hold the rest of the design fixed, add or remove only that objective step, then evaluate the relevant primary metric: SFT pass rate for instruction behavior and blinded preference win rate for DPO.

The following one-prompt comparison answers a separate architecture question: can supplied context provide a current fact without changing model weights? It is an illustration, not an ablation result.

### Put the Prompting Question to a Fairer Test

“Who is Aria Voss?” alone mixes two questions: does the model know a private fact, and does it know how to answer directly? A three-way observation separates them:

1. **Base model, no Riverside context:** can only use knowledge already present in its weights.
2. **Base model, fact supplied in the prompt:** tests whether prompting can provide current evidence without changing weights.
3. **SFT adapter, no supplied fact:** tests the learned answer contract, but still cannot guarantee that an unsupported catalog claim is true.

This remains one example, not an evaluation suite. Its purpose is to clarify the roles of prompting and fine-tuning before Riverside chooses a production architecture.

In [ ]:
# One observation that separates unavailable facts from answer behavior.
zero_shot_prompt = "Who is Aria Voss?"
provided_context = (
    "Riverside context: Aria Voss serves aboard the Meridian's Promise and "
    "investigates a signal counting itself out in prime numbers.\n\n"
    "Based only on that context, who is Aria Voss?"
)

print("=== Base model, no Riverside context ===")
print(generate(base_model, zero_shot_prompt, use_chat_template=True), "\n")

print("=== Base model, Riverside fact supplied in the prompt ===")
print(generate(base_model, provided_context, use_chat_template=True), "\n")

print("=== SFT adapter, no Riverside context supplied ===")
print(generate(instruct_lora_model, zero_shot_prompt, use_chat_template=True))

print(
    "\nRead this as a role check: supplied context can provide current facts; "
    "SFT can change response behavior; neither single output proves reliable factual recall."
)

## What This Fine-Tuning Arc Established

### Evidence Checkpoint Before Decision

The notebook began with a continued-pretraining score, marked the conditions that make that score comparable, then changed the primary evidence for SFT and DPO before defining a matched parameter study. Pause before selecting a workload path and separate implementation from evidence.

### Demonstrated in this arc

| Area | What was built or measured | What that evidence supports |
| --- | --- | --- |
| Continued pretraining | Full-FT, partial-freeze, and LoRA artifacts | Each recipe can adapt a causal LM to manuscript prose |
| Instruction tuning | SFT LoRA on prompt/completion pairs | The recipe targets instruction-formatted behavior |
| Preference alignment | DPO continued from SFT | The mechanics run; the small preference experiment remains inconclusive |
| QLoRA and quantization | QLoRA mechanics plus CPU dynamic quantization | A scaling/deployment path, not a trained QLoRA quality result |
| Shared generations | Six model objects under common prompts | Concrete behavior and failure-mode hypotheses |
| Phrase diagnostic | Fixed continuations scored token by token | Selected phrases became more or less surprising after adaptation |
| Corpus probe | Common later-chapter sample scored by every candidate | Descriptive fit to that sample, with known overlap with training text |
| Objective × parameter grid | Five trained combinations and four blank combinations | Which experiments exist and which comparisons remain missing |

### Not established by this arc

- A causal quality ranking among full fine-tuning, partial freezing, and LoRA.
- Clean held-out perplexity from a predeclared shared split.
- Reliable instruction pass rate across a representative task suite.
- A DPO preference win-rate improvement over SFT.
- Factual support from cited sources, safety behavior, confidence reliability, serving latency, or end-to-end cost.

### Minimum follow-up experiment

1. Freeze train, validation, and test file lists before training.
2. Feed full FT, partial freezing, and LoRA the same ordered examples and token budget.
3. Run at least three seeds and report uncertainty.
4. Evaluate each workload with its own quality measure plus safety, latency, and cost.
5. Generate the release scorecard from those versioned results.

The evidence is now sufficient to choose **what should be evaluated next**, not to name a universal winner. The workload section turns that boundary into a Riverside decision.

> Technique references: [LoRA](https://arxiv.org/abs/2106.09685), [InstructGPT / RLHF](https://arxiv.org/abs/2203.02155), [DPO](https://arxiv.org/abs/2305.18290), and [FLAN instruction tuning](https://arxiv.org/abs/2109.01652). The [LLM evaluation arc](../05-llm-evaluation/01-llm-evaluation-metrics-and-benchmarks.ipynb) develops the remaining evaluation layers in depth.

## From Evidence to a Workload Decision

**Decision question:** after designing valid comparisons, which candidate and metric bundle belong to each Riverside use case?

![Fine-tuning decision matrix matching adaptation needs to continued pretraining, SFT plus LoRA, DPO plus LoRA, or full fine-tuning](images/finetuning-decision-matrix.png)

Use the matrix as a workload prompt, not as proof that a candidate is ready. The evidence collected in the earlier scoring, objective, and comparison sections determines which branches remain hypotheses and which are supported.

The answer begins with the workload, not with a universal model ranking:

1. Decide whether the system needs **current factual evidence** or a **persistent behavior change**.
2. If behavior must change, choose the objective that teaches it.
3. Choose the parameter strategy from a matched quality/efficiency comparison.
4. Promote only after workload-specific quality, safety, latency, and cost requirements pass.

### The Decision: What Can Riverside Hand Off Today?

The current notebook supports an architecture decision and a shortlist, not a production checkpoint winner.

| Candidate | Evolutionary role | Evidence demonstrated here | Missing before promotion | Status |
| --- | --- | --- | --- | --- |
| Baseline | Control before Riverside adaptation | Shared examples and corpus-probe score | Domain/task capability | Reject for Riverside workloads |
| Full-FT continuation | Maximum update-scope reference | Adapted prose examples | Clean test split, matched parameter comparison, operational cost | Style candidate only |
| Partial-freeze continuation | Reduce trainable state | Adapted prose examples | Same matched evidence as full FT | Style candidate only |
| LoRA continuation | Small swappable domain adapter | Adapted prose and small artifact | Same matched evidence as full FT | Style candidate only |
| SFT LoRA | Teach instruction contract efficiently | Instruction-formatted training and examples | Deterministic task suite, source support, safety, latency | Leading assistant candidate |
| DPO adapter | Add editor preference after SFT | DPO mechanics | Preference evidence on real editor labels | Do not promote |

### Workload-first handoff

| What the workload needs | Candidate route | Evidence before release |
| --- | --- | --- |
| Current, citable manuscript facts | Retrieval plus source-supported generation | Source support, safety, latency, and cost gates |
| Persistent house style | Continued-pretraining candidates | Clean prose-fit, style, safety, latency, and cost gates |
| Reliable instruction following | SFT candidate | Instruction pass rate, source support, safety, latency, and cost gates |
| A preferred version of valid SFT answers | SFT, then DPO | SFT gates plus a blinded preference gain over SFT |

Every route follows the same final rule: promote only when its required quality and operational gates pass; otherwise keep the accepted release and collect the missing evidence.

This produces three concrete handoffs:

1. **Knowledge base:** use retrieval over manuscript files and require answers supported by cited sources. Fine-tuned weights may shape behavior, but they are not the source of truth.
2. **Editing assistant:** carry SFT-LoRA forward because its objective matches instruction following. DPO becomes relevant only after a reliable SFT baseline exists and real preference labels show a repeatable gain.
3. **House-style continuation:** carry full FT, partial freezing, and LoRA as candidates, then retrain them under one matched protocol before selecting the quality/efficiency trade-off.

### Intuition to keep

- **Objective follows behavior:** continued pretraining learns a distribution, SFT learns a task contract, and DPO learns a relative preference.
- **Parameter strategy follows constraints:** full FT, freezing, LoRA, and QLoRA decide how the update is represented and paid for.
- **Metric follows workload:** prose perplexity cannot select an instruction assistant; preference win rate cannot establish factual support.
- **Examples generate hypotheses; controlled suites support decisions.**
- **Retrieval supplies current evidence; fine-tuning changes persistent behavior.**

The release section converts this shortlist into predeclared release rules, version records, a limited rollout, and a recovery plan.

## Walkthrough: Riverside's Editing Assistant

An editor supplies a manuscript passage and asks for one source-supported continuation. This is an SFT workload because the primary behavior is following that request and stopping cleanly.

The SFT-LoRA candidate advances because its objective matches the workload. Before release, it needs a held-out editing suite with instruction pass rate, source support, safety, latency, and cost gates. DPO is relevant only after the SFT baseline passes: Riverside would then add a blinded editor-preference gate against SFT.

The Aria sentence helped expose the failure modes and define this suite; it is not itself a production score. Current manuscript facts should come from the supplied passage or retrieval, not from assumed weight recall.

## Guided Walking Example: One Scenario Through the Evaluation Evolution

Use one Riverside scenario to connect the notebook's evaluation evolution end to end:

> **Scenario:** Aria Voss checks the *Meridian's Promise* status panel and discovers an anomaly. Which training recipe produces useful behavior, and what evidence would justify evaluating it further?

The cells below are intentionally grouped as a small manual lab. Run them in order after the earlier notebook cells have loaded all six candidates and defined `generate`, `continuation_logprob`, and `compute_corpus_probe`.

1. Generate fresh outputs from every candidate using the prompt contract it was trained to receive.
2. Inspect those outputs with one editable rubric rather than selecting the most fluent paragraph by instinct.
3. Hold candidate phrases fixed and measure probability shifts for the comparable continuation models.
4. Broaden from phrases to a bounded same-novel corpus probe, then combine the evidence without declaring a causal winner.

Generation remains stochastic. Change `WALKING_SAMPLE_SEED` or rerun with a new value to see whether an impression survives another sample.

> **Comparison boundary:** all six models belong in the behavior inspection. Phrase probabilities and prose perplexity are compared only across the baseline and continued-pretraining family because SFT and DPO target different behaviors. The corpus sample remains descriptive rather than clean held-out evidence.

In [ ]:
# Walking example, step 1: generate fresh evidence from every candidate.
WALKING_SAMPLE_SEED = 2026
WALKING_PROMPT = "Aria Voss checked the Meridian's Promise status panel and"

walking_candidates = {
    "Baseline": {
        "model": base_model,
        "objective": "Original pretraining",
        "use_chat_template": False,
    },
    "Full-FT continuation": {
        "model": non_instruct_ckpt,
        "objective": "Continued pretraining",
        "use_chat_template": False,
    },
    "Partial-freeze continuation": {
        "model": freeze_model,
        "objective": "Continued pretraining",
        "use_chat_template": False,
    },
    "LoRA continuation": {
        "model": lora_pt_model,
        "objective": "Continued pretraining",
        "use_chat_template": False,
    },
    "SFT LoRA": {
        "model": instruct_lora_model,
        "objective": "Supervised instruction tuning",
        "use_chat_template": True,
    },
    "DPO adapter": {
        "model": policy_model,
        "objective": "Preference optimization after SFT",
        "use_chat_template": True,
    },
}

walking_outputs = {}
for candidate_name, candidate in walking_candidates.items():
    # Reuse one seed so differences are not caused by assigning easier random streams to some models.
    torch.manual_seed(WALKING_SAMPLE_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(WALKING_SAMPLE_SEED)

    uses_chat = candidate["use_chat_template"]
    effective_prompt = instruction_prompt(WALKING_PROMPT) if uses_chat else WALKING_PROMPT
    completion = generate(
        candidate["model"],
        effective_prompt,
        max_new_tokens=80,
        use_chat_template=uses_chat,
    )
    walking_outputs[candidate_name] = {
        "objective": candidate["objective"],
        "input_contract": "SmolLM2 chat template" if uses_chat else "plain continuation",
        "effective_prompt": effective_prompt,
        "completion": completion,
    }

    print("=" * 88)
    print(f"Candidate      : {candidate_name}")
    print(f"Objective      : {candidate['objective']}")
    print(f"Input contract : {walking_outputs[candidate_name]['input_contract']}")
    print(f"Effective input: {effective_prompt!r}")
    print(f"Output         : {completion}")

print("\nRerun with a different WALKING_SAMPLE_SEED to test whether an impression persists.")

In [ ]:
# Walking example, step 2: inspect the live outputs and enter manual evidence.
WALKING_RUBRIC = {
    "catalog_specificity": "0=generic, 1=mentions Riverside details, 2=uses details coherently",
    "role_fulfillment": "0=wrong behavior, 1=partly fulfills its role, 2=clear bounded continuation/answer",
    "coherence": "0=contradictory, 1=mostly coherent, 2=coherent across the full completion",
    "unsupported_claim_risk": "0=high risk, 1=uncertain, 2=no unsupported catalog claim observed",
}

# Replace None with 0, 1, or 2 after reading the generated outputs above.
# Keep notes concrete: quote the phrase that earned or lost a point.
walking_manual_scores = {
    candidate_name: {
        "catalog_specificity": None,
        "role_fulfillment": None,
        "coherence": None,
        "unsupported_claim_risk": None,
        "notes": "",
    }
    for candidate_name in walking_outputs
}

# After comparing the two chat candidates, set this to "SFT LoRA", "DPO adapter", or "tie".
# One choice is an observation for this sample, not a preference win-rate estimate.
walking_preference_choice = None

print("MANUAL INSPECTION RUBRIC")
print("=" * 88)
for dimension, definition in WALKING_RUBRIC.items():
    print(f"{dimension:24} {definition}")

print("\nOUTPUT WORKSHEET")
print("=" * 88)
for candidate_name, evidence in walking_outputs.items():
    print(f"\n[{candidate_name}] {evidence['input_contract']}")
    print(evidence["completion"])
    print("Scores:", walking_manual_scores[candidate_name])

print("\nEdit walking_manual_scores in this cell, rerun it, then continue.")
print("For a DPO claim, also record walking_preference_choice after comparing SFT LoRA with DPO adapter.")

In [ ]:
# Walking example, step 3: hold phrases fixed and compare nested probability shifts.
walking_continuation_models = {
    "Baseline": base_model,
    "Full-FT continuation": non_instruct_ckpt,
    "Partial-freeze continuation": freeze_model,
    "LoRA continuation": lora_pt_model,
}
walking_phrases = {
    "Keeper maintenance logs": ("catalog", " opened the Keeper's maintenance logs"),
    "quantum fold drive": ("catalog", " checked the quantum fold drive"),
    "containment anomaly": ("catalog", " detected a containment-field anomaly"),
    "looked at the screen": ("generic control", " looked at the screen"),
    "went back to work": ("generic control", " went back to work"),
}

walking_phrase_results = {}
for candidate_name, model in walking_continuation_models.items():
    candidate_results = {}
    for phrase_name, (phrase_type, continuation) in walking_phrases.items():
        score = continuation_logprob(model, WALKING_PROMPT, continuation)
        candidate_results[phrase_name] = {
            "type": phrase_type,
            "mean_logprob": score["mean"],
            "tokens": score["tokens"],
        }
    walking_phrase_results[candidate_name] = candidate_results

baseline_phrase_scores = walking_phrase_results["Baseline"]
walking_phrase_summary = {}
print(f"Fixed prompt: {WALKING_PROMPT!r}")
print(
    f"{'Candidate':28} {'Catalog Δ(A-B)':>15} {'Control Δ(A-B)':>15} "
    f"{'Selectivity Δ':>15}"
)
print("-" * 78)
for candidate_name, candidate_results in walking_phrase_results.items():
    catalog_deltas = [
        row["mean_logprob"] - baseline_phrase_scores[phrase_name]["mean_logprob"]
        for phrase_name, row in candidate_results.items()
        if row["type"] == "catalog"
    ]
    control_deltas = [
        row["mean_logprob"] - baseline_phrase_scores[phrase_name]["mean_logprob"]
        for phrase_name, row in candidate_results.items()
        if row["type"] == "generic control"
    ]
    mean_catalog_shift = float(np.mean(catalog_deltas))
    mean_control_shift = float(np.mean(control_deltas))
    selectivity_contrast = mean_catalog_shift - mean_control_shift
    walking_phrase_summary[candidate_name] = {
        "mean_catalog_shift": mean_catalog_shift,
        "mean_control_shift": mean_control_shift,
        "selectivity_contrast": selectivity_contrast,
    }
    print(
        f"{candidate_name:28} {mean_catalog_shift:>+15.3f} "
        f"{mean_control_shift:>+15.3f} {selectivity_contrast:>+15.3f}"
    )

print("\nΔ(A-B) means candidate score minus baseline-model score for the same phrase.")
print(
    "Selectivity Δ means mean catalog shift minus mean generic-control shift. "
    "Positive values support a local domain-selectivity hypothesis, not a quality claim."
)

In [ ]:
# Walking example, step 4: broaden to same-novel prose and assemble the evidence ledger.
WALKING_MAX_PROBE_PARAGRAPHS = 12
walking_probe_files = [
    path for path in probe_files if path.parent.name == NOVELS["scifi"]
]
walking_probe_paragraphs = []
for path in walking_probe_files:
    for paragraph in path.read_text(encoding="utf-8").split("\n\n"):
        paragraph = paragraph.strip().replace("\n", " ")
        if len(paragraph) >= 200:
            walking_probe_paragraphs.append(paragraph)
walking_probe_paragraphs = walking_probe_paragraphs[:WALKING_MAX_PROBE_PARAGRAPHS]

if not walking_probe_paragraphs:
    raise ValueError("No same-novel probe paragraphs were found; run the earlier corpus-loader cells first.")

walking_probe_results = {}
print(
    f"Same-novel descriptive probe: {len(walking_probe_paragraphs)} paragraphs "
    f"from {len(walking_probe_files)} file(s)"
)
for candidate_name, model in walking_continuation_models.items():
    result = compute_corpus_probe(model, walking_probe_paragraphs)
    walking_probe_results[candidate_name] = result
    print(
        f"  {candidate_name:28} "
        f"PPL={result['perplexity']:8.2f}  tokens={result['tokens']:,}"
    )


def walking_manual_total(candidate_scores):
    """Return a rubric total only after the reader has filled every score."""
    dimensions = [candidate_scores[name] for name in WALKING_RUBRIC]
    return None if any(value is None for value in dimensions) else sum(dimensions)


print("\nCOMBINED WALKING-EXAMPLE EVIDENCE")
print("=" * 132)
print(
    f"{'Candidate':28} {'Manual /8':>10} {'Catalog Δ(A-B)':>15} "
    f"{'Control Δ(A-B)':>15} {'Selectivity Δ':>15} {'Probe PPL':>12}"
)
print("-" * 132)
for candidate_name in walking_candidates:
    manual_total = walking_manual_total(walking_manual_scores[candidate_name])
    phrase_summary = walking_phrase_summary.get(candidate_name)
    probe_summary = walking_probe_results.get(candidate_name)
    manual_display = "pending" if manual_total is None else str(manual_total)
    catalog_display = (
        "n/a" if phrase_summary is None else f"{phrase_summary['mean_catalog_shift']:+.3f}"
    )
    control_display = (
        "n/a" if phrase_summary is None else f"{phrase_summary['mean_control_shift']:+.3f}"
    )
    selectivity_display = (
        "n/a" if phrase_summary is None else f"{phrase_summary['selectivity_contrast']:+.3f}"
    )
    probe_display = "n/a" if probe_summary is None else f"{probe_summary['perplexity']:.2f}"
    print(
        f"{candidate_name:28} {manual_display:>10} {catalog_display:>15} "
        f"{control_display:>15} {selectivity_display:>15} {probe_display:>12}"
    )

print("\nManual SFT-vs-DPO preference for this sample:", walking_preference_choice or "pending")
print("\nInterpretation prompts:")
print("1. Which live output created the hypothesis that deserves a larger task suite?")
print("2. Did adapted-minus-base catalog shifts exceed generic-control shifts?")
print("3. Does same-novel perplexity support that direction, while remaining contaminated and non-causal?")
print("4. Which matched study and workload requirement would be required before promotion?")
print("\nDo not rank SFT or DPO by prose perplexity: their objectives require instruction and preference suites.")

## Release Evaluation: Turn Metric Values into Gates

A release gate is a value chosen **before** inspecting results. It says how a metric will be used: a quality floor blocks a weak candidate, a regression check protects the accepted release, and an operational limit decides whether a good candidate is feasible to run.

### Select Only the Metrics the Workload Needs

| Workload claim | Quality and regression gates | Operational gates |
| --- | --- | --- |
| House-style continuation | Clean held-out perplexity must not regress beyond the declared tolerance; style review must remain acceptable | p95 latency and request cost must fit the service budget |
| SFT editing assistant | Instruction pass rate, source support, and safety must clear their required floors | p95 latency and request cost must fit the service budget |
| DPO improvement | The SFT gates must still pass, and blinded preference win rate versus SFT must show a reliable improvement | The same latency and cost limits still apply |
| Retrieval-backed knowledge base | Answers must be supported by the supplied sources and handle insufficient evidence safely | Retrieval and generation latency must fit the service budget |

### Build the Editing-Assistant Gates from Individual Cases

For Aria's one-sentence editing request, Riverside does not begin with five dashboard numbers. It begins with one held-out request, its supplied manuscript passage, the generated response, and the measurements below. Repeating the same record across the suite produces the release values.

| Gate | One case asks | How the suite value is built | How Riverside uses the value |
| --- | --- | --- | --- |
| **Instruction pass rate** | Did the response contain exactly one sentence, stop cleanly, and follow every explicit format rule? | Count cases that satisfy **all** required rules, then divide by all cases. A partial answer is a failure for that case. | It is the primary SFT quality floor. Below the predeclared minimum, do not promote. |
| **Source support** | Is every factual claim about Aria, the ship, or the scene supported by the supplied passage, or did the model clearly say that evidence was missing? | Record each case as supported, unsupported, or correctly declined. Deterministic checks cover explicit facts; editors or calibrated judges review semantic claims. | Unsupported critical claims block promotion; the supported/declined rate shows whether the assistant is safe to trust with manuscript facts. |
| **Safety** | Does the response avoid each prohibited behavior in the editing policy, including the critical cases? | Run a versioned safety suite and record pass or fail for every test. Keep critical cases separate from the broad average. | One critical failure blocks promotion. The broader pass rate reveals residual risk but does not erase a critical failure. |
| **p95 latency** | Under target hardware, concurrency, prompt length, and output length, how long does the full request take? | Sort end-to-end request times; the p95 is the value that 95% of requests meet or beat. Include retrieval when retrieval is part of the workload. | If p95 exceeds the service limit, the candidate is not operationally feasible even when its quality is strong. |
| **Request cost** | What does this representative request consume in model, retrieval, and serving resources? | Divide measured total workload cost by completed requests, using the same traffic mix and token limits used for latency. | If the value exceeds the budget, optimize or select another candidate that still clears the quality floor. |

Quality gates answer **should this model do the work?** Operational gates answer **can this model do the work here?** Both must pass. A low-cost model cannot compensate for unsupported claims, and a high-quality model cannot be released into a service it cannot meet reliably.

There are no universal thresholds: perplexity depends on corpus and tokenizer; latency and cost depend on hardware and traffic. Riverside sets the values from its workload requirements and measures them on clean, versioned benchmark data. The contaminated corpus probe in this notebook is never a release input.

### Promote, Observe, or Roll Back

1. Evaluate the immutable candidate and the accepted release on the same workload-specific suite.
2. Block promotion when a required quality, safety, or regression gate fails. Among candidates that pass, use latency, cost, and artifact constraints to select one.
3. Record the artifact, tokenizer, dataset version, code revision, policy, and measured values in the decision manifest.
4. Send a small traffic slice to the selected candidate. If live behavior or an enabled gate degrades, route traffic back to the accepted immutable artifact.

The code below implements this policy shape for the editing-assistant workload. Its example thresholds are placeholders for real service requirements, and `RUN_PRODUCTION_DECISION = False` keeps it from treating this notebook's toy evidence as a production decision.

## Aria: A Short Release Decision

For the editing assistant, Riverside compares the SFT-LoRA candidate with the accepted assistant on a held-out editing suite. Instruction pass rate, source support, and safety are quality gates; p95 latency and request cost are feasibility gates. Perplexity and DPO preference remain disabled because this release claim does not need them.

If every enabled gate passes, Riverside records the exact artifact and benchmark results, then starts a small canary. If a critical check or live behavior degrades, traffic returns to the accepted artifact. This notebook has neither the external benchmark nor production traffic, so the release code remains disabled.

In [ ]:
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import random
from typing import Any, Mapping, Optional


RUN_PRODUCTION_DECISION = False


@dataclass(frozen=True)
class EvaluationPolicy:
    """Workload-specific promotion thresholds; replace defaults with service SLOs."""

    max_perplexity_regression_pct: Optional[float] = None
    min_instruction_pass_rate: Optional[float] = 0.95
    min_preference_win_rate: Optional[float] = None
    min_safety_pass_rate: float = 1.0
    max_p95_latency_ms: float = 1_500.0
    max_cost_per_1k_requests_usd: float = 1.00


@dataclass(frozen=True)
class ProductionDecisionConfig:
    workload: str = "editing-assistant"
    candidate_name: str = "Instruction-tuned (LoRA)"
    candidate_artifact: Path = Path("./checkpoints/instruction-lora")
    rollback_name: str = "previous-production"
    rollback_artifact: Path = Path("./artifacts/production/current")
    benchmark_metrics: Path = Path("./artifacts/production-benchmarks.json")
    registry_dir: Path = Path("./artifacts/finetuning-decisions")
    seed: int = 42
    policy: EvaluationPolicy = EvaluationPolicy()


def set_reproducible_seed(seed: int) -> None:
    """Seed the random sources used by this notebook's PyTorch workflow."""
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_benchmark_metrics(path: Path) -> dict[str, Any]:
    """Load offline quality, safety, latency, and cost measurements."""
    return json.loads(path.read_text(encoding="utf-8"))


def sha256_artifact(path: Path) -> str:
    """Create one deterministic digest for a checkpoint file or directory."""
    digest = hashlib.sha256()
    files = [path] if path.is_file() else sorted(item for item in path.rglob("*") if item.is_file())
    for file_path in files:
        relative_path = file_path.name if path.is_file() else file_path.relative_to(path).as_posix()
        digest.update(relative_path.encode("utf-8"))
        with file_path.open("rb") as artifact_file:
            for chunk in iter(lambda: artifact_file.read(1024 * 1024), b""):
                digest.update(chunk)
    return digest.hexdigest()


def evaluate_release(
    candidate: Mapping[str, float],
    baseline: Mapping[str, float],
    policy: EvaluationPolicy,
) -> dict[str, bool]:
    """Apply only the gates configured for this workload."""
    gates = {
        "instruction": (
            policy.min_instruction_pass_rate is None
            or candidate["instruction_pass_rate"] >= policy.min_instruction_pass_rate
        ),
        "preference": (
            policy.min_preference_win_rate is None
            or candidate["preference_win_rate"] >= policy.min_preference_win_rate
        ),
        "safety": candidate["safety_pass_rate"] >= policy.min_safety_pass_rate,
        "latency": candidate["p95_latency_ms"] <= policy.max_p95_latency_ms,
        "cost": candidate["cost_per_1k_requests_usd"] <= policy.max_cost_per_1k_requests_usd,
    }
    if policy.max_perplexity_regression_pct is not None:
        allowed = baseline["heldout_perplexity"] * (
            1.0 + policy.max_perplexity_regression_pct / 100.0
        )
        gates["perplexity"] = candidate["heldout_perplexity"] <= allowed
    return gates


def build_decision_manifest(
    config: ProductionDecisionConfig,
    benchmark: Mapping[str, Any],
    gates: Mapping[str, bool],
    artifact_digest: str,
) -> dict[str, Any]:
    """Capture the evidence and lineage needed to reproduce or roll back a release."""
    promoted = all(gates.values())
    return {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "workload": config.workload,
        "decision": "promote" if promoted else "rollback",
        "selected_name": config.candidate_name if promoted else config.rollback_name,
        "selected_artifact": str(
            config.candidate_artifact if promoted else config.rollback_artifact
        ),
        "candidate": {
            "name": config.candidate_name,
            "artifact": str(config.candidate_artifact),
            "sha256": artifact_digest,
        },
        "rollback": {
            "name": config.rollback_name,
            "artifact": str(config.rollback_artifact),
        },
        "reproducibility": {
            "seed": config.seed,
            "dataset_fingerprint": benchmark["dataset_fingerprint"],
            "code_revision": benchmark["code_revision"],
            "base_model": MODEL_NAME,
        },
        "policy": asdict(config.policy),
        "metrics": benchmark["candidate"],
        "baseline_metrics": benchmark["baseline"],
        "gates": dict(gates),
    }


def write_decision_manifest(manifest: Mapping[str, Any], registry_dir: Path) -> Path:
    """Write an immutable, content-addressed decision record."""
    registry_dir.mkdir(parents=True, exist_ok=True)
    canonical = json.dumps(manifest, sort_keys=True, separators=(",", ":"))
    decision_id = hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:12]
    output_path = registry_dir / f"decision-{decision_id}.json"
    output_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
    return output_path

In [ ]:
production_config = ProductionDecisionConfig()

if RUN_PRODUCTION_DECISION:
    set_reproducible_seed(production_config.seed)

    # Release metrics must come from the external benchmark artifact. The shared corpus probe in this
    # notebook is training-contaminated and is intentionally never copied into a production gate.
    benchmark = load_benchmark_metrics(production_config.benchmark_metrics)
    candidate_metrics = dict(benchmark["candidate"])
    benchmark = {**benchmark, "candidate": candidate_metrics}
    gates = evaluate_release(
        candidate_metrics,
        benchmark["baseline"],
        production_config.policy,
    )

    if not production_config.candidate_artifact.exists():
        raise FileNotFoundError(
            f"Candidate artifact not found: {production_config.candidate_artifact}"
        )

    artifact_digest = sha256_artifact(production_config.candidate_artifact)
    manifest = build_decision_manifest(
        production_config,
        benchmark,
        gates,
        artifact_digest,
    )
    manifest_path = write_decision_manifest(manifest, production_config.registry_dir)

    for gate_name, passed in gates.items():
        print(f"{gate_name:>12}: {'PASS' if passed else 'FAIL'}")
    print(f"Decision: {manifest['decision'].upper()} -> {manifest['selected_name']}")
    print(f"Manifest: {manifest_path}")
else:
    print(
        "Production decision workflow is disabled. Set RUN_PRODUCTION_DECISION = True "
        "only after supplying clean benchmark metrics and an explicit rollback artifact."
    )